# 04 — Small SAR Scan

## What this notebook does
Takes a seed peptide, generates all single amino-acid substitution variants
(or a filtered subset), scores them all on the heuristic lane, and produces a
ranked table and a substitution heatmap.

## What decision it helps make
> "Which positions and substitutions in my seed peptide appear most promising
> under the heuristic lane? Which variants should I consider for promotion?"

## What it cannot prove
- That higher-ranked variants actually bind better
- Fine-grained rank ordering within a narrow score band
- Anything about selectivity, stability, or activity

---

> **What is SAR?**
> SAR stands for **Structure-Activity Relationship**. In practice here it means:
> we systematically substitute one residue at a time in a seed sequence and
> observe how the score changes. Positions where certain substitutions consistently
> increase or decrease the score are 'sensitive' positions — they may matter
> biologically (or may just be artefacts of the heuristic lane).
>
> **Important:** this notebook uses the same heuristic lane as notebooks 02–03.
> It has the same limitations. The heatmap shows *relative ordering within this lane*,
> not binding data.

> **Prerequisite:** run notebook 02 first and confirm panel status is not 'fail'.

## Free vs Paid Colab

This is the most compute-sensitive notebook in the cookbook.
The heuristic lane itself is fast (all Python, no GPU), but the panel size
can grow quickly with full single-mutant scans.

| Config | Free | Paid |
|--------|------|------|
| `COMPUTE_TIER` | `"free"` | `"paid"` |
| Max variants | 40 | 200 |
| Positions scanned | First N positions | All positions |
| Substitution alphabet | 16 amino acids | All 20 |
| Heatmap | Small (cropped) | Full |

A 12-residue seed scanned with 19 substitutions per position = 228 variants.
On free Colab the cap of 40 means you scan roughly 2 positions fully.
Set `SCAN_POSITIONS` explicitly to choose which positions matter most.

The heuristic lane runs in < 1 second per peptide, so 200 variants takes a few seconds.
The bottleneck on free Colab is RAM for large plots, not compute.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "biopython", "matplotlib"])
print("Dependencies ready.")

In [ ]:
import sys, pathlib

# ── Environment detection ─────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    for _p in [
        pathlib.Path('/content/cookbooks/colab-basics'),
        pathlib.Path('/content/colab-basics'),
    ]:
        if _p.exists():
            COOKBOOK_DIR = _p
            break
    else:
        raise RuntimeError(
            "Cookbook not found. Clone the repo first:\n"
            "  !git clone https://github.com/peptidemodel/cookbooks /content/cookbooks"
        )
    WORKSPACE_DIR = pathlib.Path('/content/workspace')
else:
    COOKBOOK_DIR = pathlib.Path('..').resolve()
    WORKSPACE_DIR = COOKBOOK_DIR / 'workspace'

# ============================================================
# CONFIGURATION
# ============================================================

# Compute tier — controls scan size cap only. No GPU lane in this cookbook.
COMPUTE_TIER = "free"  # "free" or "paid"

SEED_PEPTIDE = "RIEGTKLNRSFM"  # replace with your candidate or positive reference
SEED_LABEL   = "seed_v1"

# Positions to scan (0-based). None = all positions up to MAX_VARIANTS cap.
SCAN_POSITIONS = None

# Substitution alphabet.
# Free tier excludes C, M, H, P to keep variant counts manageable.
# C and M are biologically important (disulfides, methionine chemistry) — if you want
# to scan them, add them back here or switch to COMPUTE_TIER = "paid".
if COMPUTE_TIER == "paid":
    SUBSTITUTION_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"
    MAX_VARIANTS = 200
else:
    SUBSTITUTION_ALPHABET = "ADEFGIKLNQRSTVWY"  # 16 AAs; excludes C M H P
    MAX_VARIANTS = 40

IMPROVE_THRESHOLD_FRACTION = 1.05  # flag variants ≥ 5% above seed score

TARGET_SPEC_PATH = WORKSPACE_DIR / "target_spec.json"
OUTPUT_DIR       = WORKSPACE_DIR / "sar_scan"

print(f"Seed: {SEED_LABEL} = {SEED_PEPTIDE} ({len(SEED_PEPTIDE)} aa)")
print(f"Tier: {COMPUTE_TIER}, max variants: {MAX_VARIANTS}, alphabet: {SUBSTITUTION_ALPHABET}")
full_size = len(SEED_PEPTIDE) * (len(SUBSTITUTION_ALPHABET) - 1)
print(f"Full single-mutant scan = {full_size} variants — capped at {MAX_VARIANTS}")

In [ ]:
# Setup
import sys, json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(COOKBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(COOKBOOK_DIR))

from shared.target_utils import load_target_spec
from shared.scoring_utils import score_panel, score_peptide
from shared.panel_utils import generate_single_mutant_variants, save_panel_results

if not TARGET_SPEC_PATH.exists():
    raise FileNotFoundError(f"Target spec not found at {TARGET_SPEC_PATH}. Run notebook 01 first.")

target_spec = load_target_spec(TARGET_SPEC_PATH)
print(f"Target: {target_spec['pdb_id']} chain {target_spec['chain_id']}")

## Step 1 — Score the Seed

First establish the baseline: how does the seed peptide score on this lane?
All variant scores will be interpreted relative to this baseline.

In [ ]:
seed_result = score_peptide(SEED_PEPTIDE, target_spec)
SEED_SCORE = seed_result['composite_score']
IMPROVE_THRESHOLD = SEED_SCORE * IMPROVE_THRESHOLD_FRACTION

print(f"Seed peptide: {SEED_PEPTIDE}")
print(f"Seed score:   {SEED_SCORE:.4f}")
print(f"Improve threshold (>{IMPROVE_THRESHOLD_FRACTION:.0%} of seed): {IMPROVE_THRESHOLD:.4f}")
print(f"\nSub-scores:")
for k, v in seed_result['sub_scores'].items():
    print(f"  {k:<25}: {v}")

## Step 2 — Generate Single-Mutant Variants

In [ ]:
variants = generate_single_mutant_variants(
    seed=SEED_PEPTIDE,
    positions=SCAN_POSITIONS,
    substitutions=SUBSTITUTION_ALPHABET,
    max_variants=MAX_VARIANTS,
)

print(f"Generated {len(variants)} variants.")
if len(variants) >= MAX_VARIANTS:
    print(f"NOTE: Reached the {MAX_VARIANTS}-variant cap. Not all positions were scanned.")
    print("(Each position is always completed fully before stopping.)")
    positions_covered = sorted(set(v['position'] for v in variants))
    print(f"Positions covered (0-based): {positions_covered}")

# Preview first few
print("\nFirst 5 variants:")
for v in variants[:5]:
    print(f"  {v['label']:<20} {v['sequence']}")

## Step 3 — Score All Variants

In [ ]:
print(f"Scoring {len(variants)} variants...")
scored = score_panel(variants, target_spec)
print("Done.")

# Annotate with delta vs seed and improved flag
for r in scored:
    r['delta_vs_seed'] = round(float(r.get('composite_score', 0)) - SEED_SCORE, 4)
    r['improved'] = float(r.get('composite_score', 0)) >= IMPROVE_THRESHOLD

n_improved = sum(1 for r in scored if r['improved'])
print(f"\nVariants above improve threshold ({IMPROVE_THRESHOLD:.4f}): {n_improved}")

print(f"\nTop 10 variants:")
print(f"{'Label':<22} {'Score':>7} {'Delta':>8} {'Improved':<10} {'Sequence'}")
print("-" * 75)
for r in scored[:10]:
    imp = "YES" if r['improved'] else ""
    print(f"{r['label']:<22} {float(r.get('composite_score', 0)):>7.4f} "
          f"{r['delta_vs_seed']:>+8.4f} {imp:<10} {r['sequence']}")

## Step 4 — Substitution Heatmap

The heatmap shows the score change (delta vs seed) for each amino acid substitution
at each position. Green = improvement over seed; red = degradation.

Positions with large deltas (either direction) are 'sensitive' positions where
the heuristic lane detects meaningful property differences.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Build position x substitution matrix
positions_in_scan = sorted(set(v['position'] for v in scored if 'position' in v))
aas_in_scan = sorted(set(v['new_aa'] for v in scored if 'new_aa' in v))

# delta matrix: shape (n_positions, n_substitutions)
matrix = np.zeros((len(positions_in_scan), len(aas_in_scan)))
for r in scored:
    if 'position' not in r or 'new_aa' not in r:
        continue
    pi = positions_in_scan.index(r['position'])
    ai = aas_in_scan.index(r['new_aa'])
    matrix[pi, ai] = r['delta_vs_seed']

n_pos = len(positions_in_scan)
fig_h = max(3, n_pos * 0.45)
fig, ax = plt.subplots(figsize=(max(8, len(aas_in_scan) * 0.5), fig_h))

vmax = max(0.01, np.abs(matrix).max())
im = ax.imshow(matrix, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')

ax.set_xticks(range(len(aas_in_scan)))
ax.set_xticklabels(aas_in_scan, fontsize=9)
ax.set_yticks(range(len(positions_in_scan)))
ax.set_yticklabels(
    [f"pos{p+1} ({SEED_PEPTIDE[p] if p < len(SEED_PEPTIDE) else '?'})" for p in positions_in_scan],
    fontsize=9
)
ax.set_xlabel("Substituted amino acid", fontsize=10)
ax.set_ylabel("Position (original AA)", fontsize=10)
ax.set_title(
    f"SAR Heatmap — Delta score vs seed\n"
    f"Seed: {SEED_PEPTIDE} (score={SEED_SCORE:.3f}) | "
    f"{target_spec['pdb_id']} chain {target_spec['chain_id']}",
    fontsize=9
)

plt.colorbar(im, ax=ax, label="Delta score vs seed", shrink=0.8)
plt.tight_layout()

heatmap_path = OUTPUT_DIR / "sar_heatmap.png"
plt.savefig(heatmap_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Heatmap saved to {heatmap_path}")

## Step 5 — Save Outputs

In [ ]:
# Save full ranked table
save_panel_results(scored, OUTPUT_DIR / "sar_scores.csv")
print(f"Full ranked table saved to {OUTPUT_DIR / 'sar_scores.csv'}")

# Save shortlist (improved variants only)
improved = [r for r in scored if r.get('improved')]
if improved:
    save_panel_results(improved, OUTPUT_DIR / "sar_shortlist.csv")
    print(f"Shortlist ({len(improved)} improved variants) saved to {OUTPUT_DIR / 'sar_shortlist.csv'}")
else:
    print("No variants met the improve threshold. Consider lowering IMPROVE_THRESHOLD_FRACTION.")

# Save scan metadata
import json as _json
scan_meta = {
    "seed_label": SEED_LABEL,
    "seed_sequence": SEED_PEPTIDE,
    "seed_score": SEED_SCORE,
    "compute_tier": COMPUTE_TIER,
    "n_variants_scored": len(scored),
    "n_improved": n_improved,
    "improve_threshold": IMPROVE_THRESHOLD,
    "improve_threshold_fraction": IMPROVE_THRESHOLD_FRACTION,
    "positions_scanned": positions_in_scan,
    "substitution_alphabet": SUBSTITUTION_ALPHABET,
    "top5_variants": [
        {"label": r['label'], "sequence": r['sequence'],
         "score": r.get('composite_score'), "delta": r.get('delta_vs_seed')}
        for r in scored[:5]
    ],
    "lane": "heuristic_composition",
    "caveats": [
        "Heuristic lane — relative ordering only, not binding predictions.",
        "Top variants should be confirmed with a stronger lane before promotion.",
    ],
}
(OUTPUT_DIR / "sar_scan_meta.json").write_text(_json.dumps(scan_meta, indent=2))
print(f"Scan metadata saved.")

## Interpreting the Heatmap

**Green cells** (positive delta): this substitution scores higher than the seed.
Worth considering — but remember this is a heuristic lane.

**Red cells** (negative delta): this substitution scores lower. May indicate a
position important for the pocket properties, or just a property mismatch.

**White/near-zero cells**: the substitution makes little difference. Either the
position is not sensitive in this lane, or the heuristic cannot resolve the difference.

### What to do next

1. Pick 2–5 top-ranked variants from `sar_shortlist.csv`
2. Check that they are chemically sensible (not just alanine substitutions everywhere)
3. Run notebook 05 to promote them with a documented rationale

Do not automatically trust the rank order within the top 5–10. The heuristic lane
cannot reliably distinguish sequences that are close in score.

## Outputs

| File | Description |
|------|-------------|
| `workspace/sar_scan/sar_scores.csv` | All variants, ranked by score |
| `workspace/sar_scan/sar_shortlist.csv` | Improved variants only |
| `workspace/sar_scan/sar_heatmap.png` | Substitution heatmap |
| `workspace/sar_scan/sar_scan_meta.json` | Scan metadata and top 5 summary |

## Next Notebook

→ **05_promote_to_strong_validation.ipynb**

That notebook collects your shortlisted candidates and builds a promotion bundle
for handoff to a stronger validation lane.